# State Estimation and Target Tracking Tutorial

Welcome to the **kalbee** target tracking tutorial! In this notebook, we will demonstrate:
1. Simulating a maneuvering target in 2D space.
2. Tracking it using a Constant Velocity (CV) Kalman Filter, a Constant Acceleration (CA) Kalman Filter, and an Interacting Multiple Model (IMM) Filter.
3. Comparing their tracking performance and visualizing model probability transitions.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from kalbee import KalmanFilter, InteractingMultipleModel
from kalbee.modules.utils.metrics import rmse

## 1. Simulate Maneuvering Target

We generate a 2D trajectory of a target that starts with a constant velocity, executes a sharp acceleration turn, and returns to constant velocity.

In [ ]:
dt = 0.1
t = np.arange(0, 15, dt)
T = len(t)

# Trajectory generation
true_x, true_y = [], []
px, py = 0.0, 0.0
vx, vy = 2.0, 1.5

for step in t:
    if 5.0 <= step < 10.0:
        ax, ay = 1.0, -1.2  # Maneuver acceleration
    else:
        ax, ay = 0.0, 0.0

    vx += ax * dt
    vy += ay * dt
    px += vx * dt + 0.5 * ax * dt**2
    py += vy * dt + 0.5 * ay * dt**2

    true_x.append(px)
    true_y.append(py)

true_x = np.array(true_x)
true_y = np.array(true_y)

# Add noise to simulate YOLO detections
np.random.seed(42)
noise_std = 0.5
meas_x = true_x + np.random.randn(T) * noise_std
meas_y = true_y + np.random.randn(T) * noise_std

plt.figure(figsize=(10, 6))
plt.plot(true_x, true_y, "g-", label="True Trajectory")
plt.scatter(meas_x, meas_y, color="red", alpha=0.3, s=15, label="Noisy Detections")
plt.title("Simulated Maneuvering Target Path")
plt.xlabel("X position")
plt.ylabel("Y position")
plt.legend()
plt.grid(True)
plt.show()

## 2. Initialize and Run Filters

We setup a Constant Velocity (CV) filter, a Constant Acceleration (CA) filter, and an IMM blending both.

In [ ]:
# CV Model (4D State: [x, y, vx, vy])
state_cv = np.array([[0.0], [0.0], [2.0], [1.5]])
cov_cv = np.eye(4) * 2.0
F_cv = np.array(
    [
        [1.0, 0.0, dt, 0.0],
        [0.0, 1.0, 0.0, dt],
        [0.0, 0.0, 1.0, 0.0],
        [0.0, 0.0, 0.0, 1.0],
    ]
)
Q_cv = np.eye(4) * 0.05
H_cv = np.array([[1.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0]])
R = np.eye(2) * (noise_std**2)

kf_cv = KalmanFilter(state_cv.copy(), cov_cv.copy(), F_cv, Q_cv, H_cv, R.copy())

# CA Model (6D State: [x, y, vx, vy, ax, ay])
state_ca = np.array([[0.0], [0.0], [2.0], [1.5], [0.0], [0.0]])
cov_ca = np.eye(6) * 2.0
F_ca = np.array(
    [
        [1.0, 0.0, dt, 0.0, 0.5 * dt**2, 0.0],
        [0.0, 1.0, 0.0, dt, 0.0, 0.5 * dt**2],
        [0.0, 0.0, 1.0, 0.0, dt, 0.0],
        [0.0, 0.0, 0.0, 1.0, 0.0, dt],
        [0.0, 0.0, 0.0, 0.0, 1.0, 0.0],
        [0.0, 0.0, 0.0, 0.0, 0.0, 1.0],
    ]
)
Q_ca = np.eye(6) * 0.2
H_ca = np.array([[1.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0, 0.0, 0.0]])

kf_ca = KalmanFilter(state_ca.copy(), cov_ca.copy(), F_ca, Q_ca, H_ca, R.copy())

# IMM Model setup (CV and CA must share state dimensions, so we inflate CV to 6D)
F_cv_6 = np.array(
    [
        [1.0, 0.0, dt, 0.0, 0.0, 0.0],
        [0.0, 1.0, 0.0, dt, 0.0, 0.0],
        [0.0, 0.0, 1.0, 0.0, 0.0, 0.0],
        [0.0, 0.0, 0.0, 1.0, 0.0, 0.0],
        [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
        [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
    ]
)
Q_cv_6 = np.zeros((6, 6))
Q_cv_6[:4, :4] = Q_cv

kf_cv_imm = KalmanFilter(state_ca.copy(), cov_ca.copy(), F_cv_6, Q_cv_6, H_ca, R.copy())
kf_ca_imm = KalmanFilter(state_ca.copy(), cov_ca.copy(), F_ca, Q_ca, H_ca, R.copy())

model_transition = np.array([[0.95, 0.05], [0.05, 0.95]])
model_probabilities = np.array([0.8, 0.2])

imm = InteractingMultipleModel(
    [kf_cv_imm, kf_ca_imm], model_transition, model_probabilities
)

track_cv, track_ca, track_imm = [], [], []
probs_cv, probs_ca = [], []

for i in range(T):
    z = np.array([[meas_x[i]], [meas_y[i]]])

    # CV
    kf_cv.predict()
    kf_cv.update(z)
    track_cv.append((kf_cv.state[0, 0], kf_cv.state[1, 0]))

    # CA
    kf_ca.predict()
    kf_ca.update(z)
    track_ca.append((kf_ca.state[0, 0], kf_ca.state[1, 0]))

    # IMM
    imm.predict()
    imm.update(z)
    track_imm.append((imm.state[0, 0], imm.state[1, 0]))
    probs_cv.append(imm.model_probabilities[0])
    probs_ca.append(imm.model_probabilities[1])

## 3. Visualize and Compare Results

In [ ]:
track_cv = np.array(track_cv)
track_ca = np.array(track_ca)
track_imm = np.array(track_imm)

print("Position Tracking RMSE:")
print(f"CV Filter: {rmse(track_cv[:, 0], true_x):.4f}")
print(f"CA Filter: {rmse(track_ca[:, 0], true_x):.4f}")
print(f"IMM Filter: {rmse(track_imm[:, 0], true_x):.4f}")

# Plot model probability over time
plt.figure(figsize=(10, 4))
plt.plot(t, probs_cv, label="CV Model Probability", color="blue")
plt.plot(t, probs_ca, label="CA Model Probability", color="orange")
plt.axvspan(5.0, 10.0, color="red", alpha=0.1, label="Target Acceleration Phase")
plt.title("IMM Model Probability Adaption")
plt.xlabel("Time (seconds)")
plt.ylabel("Probability")
plt.legend()
plt.grid(True)
plt.show()